In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_gold_updated = spark.table(f"{catalogo}.{esquema_source}.sales_cleaned_grouped")

In [0]:
df_sales_by_category = (
    df_gold_updated
    .filter(col("SaleOrReturn") == "sale")
    .groupBy(
        col("CategoryCode"),
        col("CategoryName")
    )
    .agg(
        sum("QuantitySold_kilo").alias("TotalQuantitySold"),
        sum(
            col("QuantitySold_kilo") * col("UnitSellingPrice")
        ).alias("TotalRevenue")
    )
)

In [0]:
df_top_products = (
    df_gold_updated
    .filter(col("SaleOrReturn") == "sale")
    .groupBy(
        "ItemCode",
        "ItemName",
        "CategoryName"
    )
    .agg(
        sum("QuantitySold_kilo").alias("TotalQuantitySold"),
        sum(
            col("QuantitySold_kilo") * col("UnitSellingPrice")
        ).alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

In [0]:
df_daily_sales = (
   df_gold_updated
   .filter(col("SaleOrReturn") == "sale")
   .groupBy(
       "Date"
   )
   .agg(
       count("*").alias("NumberOfTransactions"),
       sum("QuantitySold_kilo").alias("TotalQuantitySold"),
       sum(col("QuantitySold_kilo") * col("UnitSellingPrice")).alias("TotalRevenue")
   ) 
   .orderBy(col("Date").asc())
)

In [0]:
df_sales_by_category.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.sales_by_category")

In [0]:
df_top_products.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.top_products")

In [0]:
df_daily_sales.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.daily_sales")

In [0]:
display(df_sales_by_category)

In [0]:
display(df_top_products)

In [0]:
display(df_daily_sales)